# B. 스케일링 / 변환

작업형 1유형 연습 — 포함: T1-4, T1-9, T1-10, T1-11

> 데이터는 seaborn/sklearn 내장셋(titanic, tips, penguins 등)으로 대체해 연습.
> 문제 지문은 원문 복붙 대신 **본인 말로 요약**해서 적기 (출처: dataq 체험환경 / 캐글 등).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

## 왜도/첨도 (로그스케일)

**문제 요약(본인 말로):** 

**풀이 메모:** skew(), kurt(), log1p()

In [7]:
# T1-4 왜도/첨도 (로그스케일)


df = sns.load_dataset('titanic')

## 왜도 - 데이터가 한쪽으로 얼마나 기울었는가?
print("왜도: ", df['age'].skew())



## 첨도 - 데이터 분포의 뾰족한 정도
print("첨도: ", df['age'].kurt())


## 로그 스케일 - 데이터의 왜도가 너무 클 때, 완화하기 위해 로그 사용
df['log_fare'] = np.log1p(df['fare'])

## 다시 왜도와 첨도 구하기 - log 스케일 지난 후에 대해서 진행하면 됨 (생략)




첨도:  0.38910778230082704
첨도:  0.17827415364210353


둘다 0 기준으로 본다. 
* $-0.5 \le sk \le 0.5$: 상당히 대칭적 (안정적)$-1 < sk < -0.5$ 또는 $0.5 < sk < 1$: 적당히 치우침
* $sk \le -1$ 또는 $sk \ge 1$: 심하게 치우침 ➡️ 로그 변환(log1p) 강력 권장!


  왜도 값 (sk),분포의 형태,의미
sk>0 (양수),오른쪽으로 꼬리가 긴 분포,소수의 큰 값(아주 비싼 Fare 등)이 오른쪽에 치우쳐 있음
sk=0,완벽한 좌우 대칭,정규분포 모양
sk<0 (음수),왼쪽으로 꼬리가 긴 분포,소수의 작은 값이 왼쪽에 치우쳐 있음

첨도 값 (ku)분포의 형태, 의미$ku > 0$ (양수)정규분포보다 훨씬 뾰족함데이터가 중앙에 몰려있고, 꼬리가 두꺼움 (이상치 많음)$ku = 0$정규분포와 동일함적당한 뾰족함과 꼬리 두께$ku < 0$ (음수)정규분포보다 완만함/평평함데이터가 넓게 퍼져있고, 중심부가 뭉툭함

## 표준화 (중앙값)

**문제 요약(본인 말로):** 표준화시키고 중앙값 구하라

**풀이 메모:** StandardScaler, fit_transform(), median()

In [12]:
# T1-9 표준화 (중앙값)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

##주의 하자 - StandardScaler는 2차원 데이터를 원함
df['scaled_age'] = scaler.fit_transform(df[['age']])

# 표준화한 후의 중앙값 구하기
result = df['scaled_age'].median()
print(result)




-0.11704877767623337


## Yeo-Johnson & Box-Cox (거의 안나옴)

- 왜도와 첨도를 조절하기 위해 데이터를 정규분포에 가깝게 맞춰주는 '파워 트랜스폼(Power Transform, 거듭제곱 변환)' 기법

**문제 요약(본인 말로):** 

**풀이 메모:** 

In [14]:
from sklearn.preprocessing import PowerTransformer
# 1. Box-Cox 변환 (df[['fare']] 데이터 자체에 + 1을 해줘야 합니다)
pt_box = PowerTransformer(method='box-cox')
df['box_cox_fare'] = pt_box.fit_transform(df[['fare']] + 1)  # 괄호 위치 수정!

# 2. Yeo-Johnson 변환 (0이 있어도 괜찮으니 그대로 진행)
pt_yeo = PowerTransformer(method='yeo-johnson')
df['yeo_john_fare'] = pt_yeo.fit_transform(df[['fare']])

## min-max scaling



**문제 요약(본인 말로):** min-max스케일링 기준 상하위 5% 구하기

**풀이 메모:** sklearn.preprocessing의 MinMaxScaler


In [17]:
# T1-11 min-max scaling
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df['scaled_age'] = scaler.fit_transform(df[['age']]) ### 여기도 조심! 2차원 데이터 입력!!

#하위 5% 와 상위 5% 지점 값 구하기
lower_5 = df['scaled_age'].quantile(0.05)
upper_5 = df['scaled_age'].quantile(0.95)

print("하위 5% 지점 값:", lower_5)
print("상위 5% 지점 값:", upper_5)


하위 5% 지점 값: 0.04498617743151546
상위 5% 지점 값: 0.6984166876099522


## 주의: sklearn의 모든 스케일러는 내부적으로 항상 2차원 데이터 (df 등)을 기대하고 설계되어 있음! 